# InSight | Microsoft Phi-3-mini-4k-instruct QLoRA Fine-Tuning for Jira Ticket Synthesis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/visheshaggarwal0/InSight/blob/main/notebooks/03_phi3_qlora_jira_generator.ipynb)

### Objective
Transform raw unsupervised cluster themes into engineering-ready Jira bug tickets using **Microsoft Phi-3-mini-4k-instruct** (3.8B parameter Small Language Model).

### Why Phi-3-mini + QLoRA?
1. **Compact Footprint**: 3.8B parameters quantized to 4-bit (NF4) fits into **< 6.5 GB VRAM**, running easily on Google Colab's free T4 GPU.
2. **Exceptional Reasoning**: Outperforms Llama-2-70B and matches Mixtral 8x7B on instruction following and synthetic synthesis.
3. **Structured Schema Output**: Guaranteed formatting: Title, Priority, Customer Blast Radius, Verbatim Quotes, Steps to Reproduce, and Root Cause Hypothesis.


In [ ]:
# 1. Environment & 4-bit Package Installation
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

!pip install -q bitsandbytes peft trl accelerate transformers datasets


## 2. Load Phi-3 in 4-bit NormalFloat (NF4) with BitsAndBytes


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# Configure LoRA targets
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(model, peft_config)
lora_model.print_trainable_parameters()


## 3. Dataset Prompt Engineering & Training Template
Format customer cluster data into a structured system prompt.


In [ ]:
def format_insight_prompt(cluster):
    system_msg = "You are an expert Principal Staff Reliability & Software Engineer at InSight. Synthesize customer feedback clusters into an engineering-ready Jira bug ticket."
    
    user_content = f"""Analyze the following customer incident cluster:
- Theme: {cluster['title']}
- Severity: {cluster['severity']}
- Review Volume: {cluster['count']} reviews ({cluster['neg_rate']}% negative)
- Keywords: {', '.join(cluster['keywords'])}

Sample Customer Verbatims:
1. "{cluster['verbatims'][0]}"
2. "{cluster['verbatims'][1]}"
3. "{cluster['verbatims'][2]}"

Generate a production-ready Jira ticket with Title, Severity, Impact Analysis, Steps to Reproduce, and Root Cause Hypothesis."""

    assistant_response = f"""# [JIRA BUG] {cluster['title']} - Service Degradation & Impact
**Priority**: {cluster['severity']} | **Status**: Open | **Component**: {cluster['keywords'][0].title()}

### 1. Executive Summary & Impact Analysis
Customer feedback analysis indicates an active regression in `{cluster['keywords'][0]}` affecting {cluster['count']} tracked users with a {cluster['neg_rate']}% negative sentiment rate. Relative churn risk is calculated at >2.5x.

### 2. Evidence from Customer Verbatims
- *"{cluster['verbatims'][0]}"*
- *"{cluster['verbatims'][1]}"*
- *"{cluster['verbatims'][2]}"*

### 3. Steps to Reproduce
1. Navigate to customer application on latest production build.
2. Trigger the `{cluster['keywords'][0]}` workflow.
3. Observe intermittent timeout or exception.

### 4. Root Cause Hypothesis & Engineering Action
- **Hypothesis**: Downstream API timeout or race condition under high concurrency.
- **Recommended Action**: Inspect telemetry logs for 5xx errors; implement exponential backoff retry on client."""

    return f"<|system|>\n{system_msg}<|end|>\n<|user|>\n{user_content}<|end|>\n<|assistant|>\n{assistant_response}<|end|>"

sample_cluster = {
    "title": "Payment Gateway Timeout",
    "severity": "CRITICAL",
    "count": 482,
    "neg_rate": 94.2,
    "keywords": ["payment", "checkout", "gateway", "stripe", "timeout"],
    "verbatims": [
        "Card payment spinner spins for 60 seconds then throws error 504.",
        "Unable to complete purchase during checkout.",
        "Charged twice on bank statement because checkout timed out."
    ]
}

formatted_text = format_insight_prompt(sample_cluster)
print("Sample Prompt Formatted for Phi-3 Instruct:")
print(formatted_text[:600] + "...\n[TRUNCATED]")


## 4. Inference Demonstration & Jira Ticket Generation


In [ ]:
def generate_jira_ticket(theme_title, severity, count, neg_rate, keywords, verbatims):
    cluster_obj = {
        "title": theme_title,
        "severity": severity,
        "count": count,
        "neg_rate": neg_rate,
        "keywords": keywords,
        "verbatims": verbatims
    }
    
    prompt = f"""<|system|>
You are an expert Principal Staff Reliability & Software Engineer at InSight. Synthesize customer feedback clusters into an engineering-ready Jira bug ticket.<|end|>
<|user|>
Analyze the following customer incident cluster:
- Theme: {theme_title}
- Severity: {severity}
- Review Volume: {count} reviews ({neg_rate}% negative)
- Keywords: {', '.join(keywords)}

Sample Customer Verbatims:
1. "{verbatims[0]}"
2. "{verbatims[1]}"
3. "{verbatims[2]}"

Generate a production-ready Jira ticket with Title, Severity, Impact Analysis, Steps to Reproduce, and Root Cause Hypothesis.<|end|>
<|assistant|>
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=450,
            temperature=0.2,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

# Test generation
ticket = generate_jira_ticket(
    "Battery Drainage & Thermal Throttling",
    "CRITICAL",
    640,
    91.4,
    ["battery", "heat", "discharge", "update", "shutdown"],
    [
        "Battery drops 30% in 15 minutes after firmware v2.4.",
        "Phone becomes burning hot during simple video streaming.",
        "Device suddenly died at 18% remaining battery."
    ]
)

print(ticket)
